# The Service Runtime

Notebooks 01–07 compose EpiScope by hand: you build the vector store, the
retriever, the generator, and the workflow config yourself. That is the
*explicit* API, and it is the right one for custom work.

This notebook shows the other level: `EpiScopeRuntime`, the service layer that
assembles those same collaborators from settings and exposes three verbs —
`explore`, `classify`, `precision_mine`. The FastAPI backend, the Streamlit UI,
and the CLI's corpus-backed commands all go through it, so what you learn here
is what those surfaces actually do.

## Setup

As in the other notebooks, `episcope_nb` handles the path bootstrap and supplies
the deterministic offline stand-ins.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Locate `notebooks/episcope_nb.py`, the shared helper module. This works whether
# the kernel starts in `notebooks/` or at the repository root.
_cwd = Path.cwd()
_nb_dir = next(
    (
        directory
        for candidate in [_cwd, *_cwd.parents]
        for directory in (candidate, candidate / "notebooks")
        if (directory / "episcope_nb.py").is_file()
    ),
    None,
)
if _nb_dir is None:
    raise FileNotFoundError("Could not find notebooks/episcope_nb.py")
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

# Importing the helper also puts `src/` on sys.path when this is a checkout.
import episcope_nb as nb

WORK_DIR = nb.bootstrap()
WORK_DIR

## Build A Small Corpus

The runtime is *corpus-backed*: it expects papers to be indexed and their
metadata stored already. The next three cells build that corpus with the same
pattern used in notebook 01 — two papers, a deterministic keyword embedder, a
file-backed index, and an in-memory metadata store.

In [ ]:
sample_papers = nb.sample_papers()
list(sample_papers)

The tiny embedder below keeps this notebook offline and deterministic. In real
use you would not write one: `EmbedderFactory.get_embedder(model_name)` picks a
backend from the model name, defaulting to the torch-free `fastembed` path that
ships with the base install. That first call downloads a model, which is exactly
what these notebooks avoid.

In [ ]:
# A vocabulary tuned to the data-sharing language in these two papers.
embedder = nb.TinyKeywordEmbedder(nb.SHARING_VOCABULARY)
embedder.model_name, embedder.dim

In [ ]:
STRATEGY_NAME = "runtime-demo"
INDEX_DIR = WORK_DIR / "index"
METADATA_BACKUP = WORK_DIR / "academic_db.json"

vdb, retriever = nb.build_index(INDEX_DIR, sample_papers, embedder)
academic_db = nb.build_academic_db(
    sample_papers, STRATEGY_NAME, backup_file=METADATA_BACKUP
)

len(vdb.get_points()), academic_db.list_docs(STRATEGY_NAME)

## Configure The Runtime

`RuntimeConfig` is a frozen dataclass holding everything the runtime needs to
build its collaborators. `RuntimeConfig.from_settings()` reads it from
`AppSettings` / environment variables — that is what `episcope serve` and
`episcope studio` do. Here it is constructed explicitly so the notebook does not
depend on your environment.

The important behaviour is the **local-first fallback**: when `qdrant_url` and
`mongo_uri` are both unset, the runtime uses a file-backed index and a JSON
metadata store instead of failing. That is what makes `episcope studio` work with
no external services. Set those two fields and the same code talks to Qdrant and
MongoDB instead — the shared/server deployment.

In [ ]:
from episcope.services import EpiScopeRuntime, RuntimeConfig

config = RuntimeConfig(
    strategy_name=STRATEGY_NAME,
    # qdrant_url / mongo_uri left as None -> local-first fallback
    index_dir=INDEX_DIR,
    metadata_backup=METADATA_BACKUP,
    workflow_top_k=4,
)
runtime = EpiScopeRuntime(config)

health = runtime.health()
{
    key: health.checks[key]
    for key in [
        "vector_backend",
        "qdrant_url_configured",
        "mongo_uri_configured",
        "llm_provider",
        "llm_api_key_configured",
    ]
}

`health()` is what the API's `/health` endpoint and the Streamlit sidebar report.
`llm_api_key_configured` will be `False` here unless you happen to have a provider
key exported — which is fine, because nothing below calls a real model.

## Explore

`explore` is retrieval plus an optional generated answer. With
`generate_answer=False` it needs no LLM at all.

Every runtime method accepts `retriever=`, `generator=`, and `academic_db=`
overrides. That is not just a testing hook — it is required here. Left to itself,
`build_retriever()` would open the file index and ask `EmbedderFactory` for a
`"tiny-keyword-demo"` embedder, which no provider can supply. Passing the
retriever we already built keeps the notebook offline and shows the seam where
you can substitute your own components.

In [ ]:
result = runtime.explore(
    "Is the dataset available in a public repository?",
    top_k=3,
    retriever=retriever,
)

print(f"{result.retrieval_count} chunk(s) for: {result.query}\n")
for chunk in result.retrieved_chunks:
    print(f"- {chunk.paper_id} | {chunk.section_title} ({chunk.similarity_score:.3f})")
    print(f"    {chunk.text[:140]}")
print("\nanswer:", result.answer)

With `generate_answer=True` the runtime calls `build_generator()`, which needs a
configured provider key. Passing a generator explicitly keeps it offline —
`NoLLMGenerator` just concatenates the retrieved text, so you can check the
provenance wiring before spending a token.

In [ ]:
from episcope.rag.generation.nollm_generator import NoLLMGenerator

answered = runtime.explore(
    "Is the dataset available in a public repository?",
    top_k=2,
    generate_answer=True,
    retriever=retriever,
    generator=NoLLMGenerator(),
)

print(answered.answer[:400])
print("\nprovenance:")
for evidence in answered.provenance.evidences:
    print(f"- {evidence.paper_id}: {evidence.section}")

## Classify And Mine

`classify` and `precision_mine` wrap the workflow classes from notebooks 05 and
07. The runtime resolves the *kind* string through the workflow registry, so
`classifier_kind="data_accessibility"` and any declarative task you have
registered are selected the same way.

The stub generator below returns fixed JSON. Note that both workflows default to
`structured_output="schema"`, so a real `LLMGenerator` receives a
`response_schema` kwarg and is constrained at decode time; the stub accepts and
ignores it via `**kwargs`.

In [ ]:
# `nb.FixedJSONGenerator` returns fixed JSON whatever the contexts. Note what it
# has to tolerate: both workflows default to `structured_output="schema"`, so the
# runtime passes a `response_schema` kwarg that a real `LLMGenerator` would use
# to constrain decoding, and the stub must accept and ignore it.
nb.FixedJSONGenerator.__doc__

In [ ]:
classification = runtime.classify(
    "paper_open_data",
    classifier_kind="data_accessibility",
    retriever=retriever,
    academic_db=academic_db,
    generator=nb.FixedJSONGenerator(
        {
            "classification": ["A"],
            "primary_label": "A",
            "reasoning": "De-identified data and code are available in a public repository.",
            "confidence": 0.92,
            "class_probabilities": {"A": 0.92, "B": 0.03, "C": 0.02, "D": 0.01, "E": 0.01, "F": 0.01},
        },
        model_id="demo-data-accessibility",
    ),
)

labels = [label.value for label in classification.decision.result.classification]
print("labels:", labels)
print("confidence:", classification.decision.result.confidence)
print("evidence chunks:", len(classification.decision.top_evidence))

In [ ]:
extraction = runtime.precision_mine(
    "paper_open_data",
    miner_kind="find_data_sources",
    retriever=retriever,
    academic_db=academic_db,
    generator=nb.FixedJSONGenerator(
        {
            "description": "One registry and one public repository.",
            "items": [
                {
                    "name": "National Hospital Registry",
                    "url": None,
                    "explanation": "Named as the source of patient records.",
                    "raw_text": "The analysis used patient records from the National Hospital Registry.",
                }
            ],
        },
        model_id="demo-find-data-sources",
    ),
)

print(extraction.result.description)
for item in extraction.result.items:
    print(f"- {item.name}: {item.explanation}")

## The Builders Underneath

The three verbs are thin. Everything they assemble is also available directly, so
you can borrow one piece without adopting the whole facade:

| method | returns | notes |
| --- | --- | --- |
| `build_retriever()` | `Retriever` | Qdrant when `qdrant_url` is set, else a `FileDB` index |
| `build_db()` | `AcademicDB` | `MongoAcademicDB` when `mongo_uri` is set, else `InMemoryAcademicDB` |
| `build_llm_client()` | `LLMClient` | provider dispatch (gemini / openai / openrouter / anthropic / ollama) |
| `build_generator()` | `LLMGenerator` | the client wrapped for workflow use |
| `build_classifier_config(kind)` | classifier config | resolved through the workflow registry |
| `build_precision_miner_config(kind)` | miner config | resolved through the workflow registry |
| `build_evidence_reranker()` | reranker or `None` | `None` unless `evidence_reranker_kind` is set |

The ones that reach outside the process (`build_retriever` with a real embedding
model, `build_llm_client`) are the ones this notebook substitutes.

In [ ]:
db = runtime.build_db()
classifier_config = runtime.build_classifier_config("data_accessibility")

print("db:", type(db).__name__)
print("reranker:", runtime.build_evidence_reranker())
print("classifier top_k:", classifier_config.top_k, "(from workflow_top_k)")
print("structured_output:", classifier_config.structured_output)
print("labels:", list(classifier_config.category_labels))

## Which Level To Use

**Reach for `EpiScopeRuntime`** when you want the same behaviour the CLI, API, and
UI give you: configuration from environment, local-first fallback, registry-based
task lookup, and one object to hold it all. It is the shortest path from
"papers are indexed" to "give me labels".

**Reach for `PaperClassifier` / `PrecisionMiner` directly** (notebooks 05 and 07)
when you are composing something the runtime does not model: a custom retriever
stack, a reranker it does not build, several configs in one process, or a test
that should construct exactly what it exercises.

They are not competing APIs — the runtime builds those same workflow objects.
Anything you can express through the facade you can also express by hand; the
reverse is not true.

Related surfaces: `episcope studio` (local-first Streamlit UI over this runtime),
`episcope serve` (the FastAPI backend), and workspaces, which give the runtime a
self-contained folder for `index/`, metadata, and `tasks/`. See the
[Python Usage](https://github.com/VinsRR/EpiScope/wiki/Python-Usage) and
[Workspaces](https://github.com/VinsRR/EpiScope/wiki/Workspaces) wiki pages.